# 10 Cost levers
Measure prompt caching on a stable prefix, then run a cheap model first and escalate on low confidence.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-10"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: a long, stable prefix

In [ ]:
import json
import pandas as pd
from kit.llm import call_llm, get_secret, pick_provider
from kit.prices import cost
from kit.rag import parse_corpus

provider = pick_provider()
pages = parse_corpus(CORPUS)
# The part that never changes goes first: instructions, then every public help page.
PREFIX = ("You are the Quillnote help assistant. Answer ONLY from the help pages below. Cite page ids like [qn-02].\n\n"
          + "\n\n".join(f"[{p['id']}] {p['title']}\n{p['body']}" for p in pages if p["visibility"] == "public"))
print("Provider:", provider)
print("Prefix:", len(PREFIX), "characters, about", len(PREFIX) // 4, "tokens. OpenAI caches prompts of 1024 tokens or more.")

Step 3: send it twice, then change one early token

In [ ]:
def ask(system, question):
    r = call_llm(question, system=system, max_output_tokens=1000)
    return {"input_tokens": r["input_tokens"], "cached_tokens": r["cached_tokens"],
            "cost_usd": round(cost(r["input_tokens"], r["output_tokens"], r["model"], r["cached_tokens"] or 0), 6),
            "latency_s": r["latency_s"]}

EDITED = PREFIX.replace("Quillnote help assistant", "Quillnote Help assistant", 1)  # one early token changed
runs = [("first send", PREFIX, "How do I export a note as PDF?"),
        ("same prefix again", PREFIX, "How do I turn on dark mode?"),
        ("one early token changed", EDITED, "How do I turn on dark mode?")]
cache_rows = [{"run": name, **ask(system, q)} for name, system, q in runs]
hit, miss = cache_rows[1]["cached_tokens"], cache_rows[2]["cached_tokens"]
if hit is None:
    print("This provider did not report cached tokens (None means unknown, not zero). See the table below.")
else:
    print(f"Same prefix again: {hit} of {cache_rows[1]['input_tokens']} input tokens came from the cache.")
    print(f"After changing one early token: {miss}. Everything after the change is a new prefix.")
pd.DataFrame(cache_rows)

Step 4: which field to read, per provider

In [ ]:
# kit/llm.py reads these fields into call_llm(...)["cached_tokens"].
pd.DataFrame([
    {"provider": "OpenAI", "field": "usage.prompt_tokens_details.cached_tokens",
     "notes": "automatic for prompts of 1024+ tokens, counted in 128 token steps"},
    {"provider": "Gemini (google-genai)", "field": "usage_metadata.cached_content_token_count",
     "notes": "implicit caching on recent models; may be missing (None), for example on the free tier"},
    {"provider": "Anthropic (the kit does not call it)", "field": "usage.cache_read_input_tokens",
     "notes": "only when the request marks blocks with cache_control"},
    {"provider": "Other OpenAI style APIs", "field": "usually the OpenAI field",
     "notes": "often missing: treat missing as unknown"},
    {"provider": "kit mock (offline)", "field": "cached_tokens in call_llm's result",
     "notes": "made up, copies the OpenAI rule"},
])

Step 5: a cascade: cheap model first, escalate on low confidence

In [ ]:
from mcp_server.server import search

MODELS = {"mock": ("mock-model-b", "mock-model"),
          "openai_compatible": ("gpt-5.4-nano", get_secret("FALLBACK_MODEL") or "gpt-5.4-mini"),
          "gemini": ("gemini-3.5-flash-lite", "gemini-3.6-flash")}
CHEAP = os.environ.get("CHEAP_MODEL") or MODELS[provider][0]
STRONG = os.environ.get("STRONG_MODEL") or MODELS[provider][1]
THRESHOLD = 0.7  # escalate when the cheap model says it is less sure than this
SYSTEM_C = ("CASCADE PROTOCOL: you are the Quillnote help assistant. Answer from the help pages and the conversation. "
            'Reply with JSON only: {"answer": "<answer, citing page ids like [qn-02]>", '
            '"confidence": <0.0 to 1.0, how sure you are the answer is right>}')

def prompt_for(messages, question):
    pages = "\n".join(f"[{h['id']}] {h['title']}: {h['snippet']}" for h in search(question, 3, CORPUS))
    convo = "\n".join(f"{m['role']}: {m['text']}" for m in messages)
    return f"Help pages:\n{pages}\n\nConversation so far:\n{convo}\n\nReply with the JSON object."

def parse(text):
    try:
        obj = json.loads(text[text.find("{"): text.rfind("}") + 1])
        return str(obj.get("answer", "")), float(obj.get("confidence", 0))
    except (ValueError, TypeError, AttributeError):
        return text.strip(), 0.0  # not JSON: count it as low confidence

print("Cheap:", CHEAP, "| strong:", STRONG, "| escalate below", THRESHOLD)

Step 6: one shared, append-only history across both models

In [ ]:
def run_cascade(turns, keep_history=True):
    history, log = [], []  # ONE list for both models. We only ever append to it.
    for n, question in enumerate(turns, 1):
        before = list(history)
        history.append({"role": "User", "text": question})
        r = call_llm(prompt_for(history, question), system=SYSTEM_C, model=CHEAP, max_output_tokens=1000)
        answer, conf = parse(r["text"])
        spent, used, sent = cost(r["input_tokens"], r["output_tokens"], r["model"]), CHEAP, None
        if conf < THRESHOLD:
            # The naive version sends the strong model only the last message.
            sent = list(history) if keep_history else history[-1:]
            r2 = call_llm(prompt_for(sent, question), system=SYSTEM_C, model=STRONG, max_output_tokens=1000)
            answer, conf2 = parse(r2["text"])
            spent += cost(r2["input_tokens"], r2["output_tokens"], r2["model"])
            used = STRONG
        history.append({"role": "Assistant", "text": answer})
        assert history[:len(before)] == before, "history was rewritten"
        log.append({"turn": n, "model": used, "escalated": sent is not None, "cheap_confidence": conf,
                    "messages_before": len(before),
                    "dropped_on_escalation": (len(before) + 1 - len(sent)) if sent is not None else 0,
                    "answer": answer[:90], "cost_usd": round(spent, 6)})
    return pd.DataFrame(log)

TURNS = ["Hi. I am on the Team plan and I look after a workspace of 12 people.",
         "How do I export a note as PDF?",
         "Given my plan, can everyone in the workspace use shared templates?"]
shared = run_cascade(TURNS, keep_history=True)
shared

Step 7: compare with an escalation that drops earlier turns

In [ ]:
naive = run_cascade(TURNS, keep_history=False)
for label, log in [("shared history", shared), ("last message only", naive)]:
    esc = log[log.escalated]
    print(f"{label}: escalated {len(esc)} of {len(log)} turns, earlier messages dropped: "
          f"{int(esc.dropped_on_escalation.sum())} | last answer: {log.answer.iloc[-1]}")

strong_only = 0.0
for n in range(1, len(TURNS) + 1):
    msgs = [{"role": "User", "text": t} for t in TURNS[:n]]
    r = call_llm(prompt_for(msgs, TURNS[n - 1]), system=SYSTEM_C, model=STRONG, max_output_tokens=1000)
    strong_only += cost(r["input_tokens"], r["output_tokens"], r["model"])
print(f"\nCost: cascade ${shared.cost_usd.sum():.6f} vs strong model on every turn ${strong_only:.6f}")